# Copresheaf Neural Network with Copresheaf Linear Transport Map

In [1]:
"""
Simple Copresheaf Neural Network Example
==========================================
A minimal example showing how to use toposheafx with a custom linear copresheaf map
on a simple simplicial complex created with toponetx.
"""
# System setup and package verification
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add the project root to path (going up 2 levels from simplicial tutorial folder)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..'))


if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added project root to path: {project_root}")

# Verify the import works
try:
    import toposheafx
    print(f"Successfully imported toposheafx from: {toposheafx.__file__}")
except ImportError as e:

import torch
import torch.nn as nn
import toponetx as tnx
from toposheafx.base.message_passing import MessagePassing



IndentationError: expected an indented block after 'except' statement on line 26 (4226279898.py, line 28)

## Create Simple Simplicial Complex for Example

In [ ]:

# Step 1: Create a simple simplicial complex using toponetx
def create_simple_simplicial_complex():
    """Create a simple triangle simplicial complex."""
    sc = tnx.SimplicialComplex()
    
    # Add simplices: 3 nodes forming a triangle
    sc.add_simplex([0, 1])  # Edge 0-1
    sc.add_simplex([1, 2])  # Edge 1-2
    sc.add_simplex([0, 2])  # Edge 0-2
    sc.add_simplex([0, 1, 2])  # Triangle
    
    return sc


## Get Adjacency Information from Simplicial Complex

In [ ]:


# Step 2: Extract edge index from simplicial complex
def get_edge_index(sc):
    """Extract directed edge index from simplicial complex."""
    edges = list(sc.skeleton(1))
    edge_list = []
    
    # Create directed edges (both directions for undirected graph)
    for edge in edges:
        edge_nodes = list(edge)
        if len(edge_nodes) == 2:
            edge_list.append([edge_nodes[0], edge_nodes[1]])
            edge_list.append([edge_nodes[1], edge_nodes[0]])
    
    return torch.tensor(edge_list, dtype=torch.long).t()



## Write Custom Linear Transport Map for Vertex Features

In [ ]:

# Step 3: Define custom linear copresheaf map factory
def custom_linear_copresheaf_map(in_dim, out_dim, edge_attr_dim):
    """
    Custom linear copresheaf morphism ρ_{y→x}: F(y) → F(x)
    
    This is a simple linear transformation that transports features
    from source node y to target node x.
    """
    # Input: [h_y, h_x, edge_attr] concatenated
    input_dim = 2 * in_dim + edge_attr_dim
    
    # Simple linear map (you can also use a lambda for even simpler cases)
    return nn.Linear(input_dim, out_dim)


# Alternative: Ultra-simple lambda-based copresheaf map
def lambda_copresheaf_map(in_dim, out_dim, edge_attr_dim):
    """Even simpler: just a linear projection using Sequential."""
    input_dim = 2 * in_dim + edge_attr_dim
    return nn.Sequential(nn.Linear(input_dim, out_dim))



## Create Copresheaf Neural Network with Copresheaf Message Passing Using Personal Linear Transport Map

In [ ]:

# Step 4: Create copresheaf layer with custom map
class SimpleCopresheafNN(nn.Module):
    """Simple copresheaf neural network for vertex feature learning."""
    
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        
        # Use toposheafx's MessagePassing with our custom copresheaf map
        self.layer1 = MessagePassing(
            in_dim=in_dim,
            out_dim=hidden_dim,
            edge_attr_dim=0,  # No edge attributes in this simple example
            aggr='sum',
            copresheaf_map_factory=custom_linear_copresheaf_map  # Our custom map!
        )
        
        self.layer2 = MessagePassing(
            in_dim=hidden_dim,
            out_dim=out_dim,
            edge_attr_dim=0,
            aggr='mean',
            copresheaf_map_factory=lambda_copresheaf_map  # Alternative lambda version
        )
    
    def forward(self, x, edge_index):
        """Forward pass through copresheaf layers."""
        x = self.layer1(x, edge_index)
        x = torch.relu(x)
        x = self.layer2(x, edge_index)
        return x



## Pass the Simplicial Complex to the Copresheaf Message Passing Neural Network 

In [ ]:

# Step 5: Run the example

print("=" * 60)
print("Simple Copresheaf Neural Network Example")
print("=" * 60)

# Create simplicial complex
print("\n1. Creating simplicial complex...")
sc = create_simple_simplicial_complex()
print(f"   Nodes: {list(sc.skeleton(0))}")
print(f"   Edges: {list(sc.skeleton(1))}")
print(f"   Triangles: {list(sc.skeleton(2))}")

# Extract edge index
print("\n2. Extracting edge index...")
edge_index = get_edge_index(sc)
print(f"   Edge index shape: {edge_index.shape}")
print(f"   Edge index:\n{edge_index}")

# Create node features
num_nodes = len(list(sc.skeleton(0)))
in_dim = 4
x = torch.randn(num_nodes, in_dim)
print(f"\n3. Node features shape: {x.shape}")
print(f"   Features:\n{x}")

# Create model with custom copresheaf map
print("\n4. Creating copresheaf model with custom linear map...")
model = SimpleCopresheafNN(in_dim=in_dim, hidden_dim=8, out_dim=2)
print(f"   Model: {model}")

# Forward pass
print("\n5. Running forward pass...")
output = model(x, edge_index)
print(f"   Output shape: {output.shape}")
print(f"   Output:\n{output}")

print("\n" + "=" * 60)
print("✓ Successfully demonstrated custom copresheaf map!")
print("=" * 60)

